# BÀI THỰC HÀNH LAB 3: LINEAR REGRESSION
**Họ và tên:** Đỗ Bá Luân  
**MSSV:** 23520895  
**Lớp:** DS102.R11.CNVN.2

---

### Assignment 1 (3 scores):

- Use the Numpy library only to construct the Linear Regression model.
- Train and Evaluate that Linear Regression model on the [Forest Fires](https://archive.ics.uci.edu/static/public/162/forest+fires.zip) dataset.

#### 1.1 Cơ sở lý thuyết:
Mô hình Hồi quy tuyến tính có dạng:
$$y = X\beta + \varepsilon$$
Nghiệm giải tích bình phương tối thiểu thông thường (OLS):
$$\hat{\beta} = (X^T X)^{-1} X^T y$$
Để ước lượng hệ số chặn $\beta_0$ (bias), ta bổ sung một cột toàn số 1 vào bên trái ma trận đặc trưng $X$:
$$X = [\mathbf{1}, X_1, X_2, ..., X_K]$$

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# 1. Đọc dữ liệu Forest Fires
data_path = 'data/forestfires.csv' if os.path.exists('data/forestfires.csv') else '../data/forestfires.csv'
df = pd.read_csv(data_path)

# 2. Ánh xạ các cột phân loại (month, day) sang giá trị số nguyên
month_map = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6, 'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}
day_map = {'mon': 1, 'tue': 2, 'wed': 3, 'thu': 4, 'fri': 5, 'sat': 6, 'sun': 7}
df['month'] = df['month'].map(month_map)
df['day'] = df['day'].map(day_map)

# 3. Phân tách đặc trưng X và nhãn y (area)
X_full = df.drop(columns=['area']).values
y_full = df['area'].values.reshape(-1, 1)

# 4. Phân chia Train/Test 80/20 với cố định random seed 42
np.random.seed(42)
indices = np.arange(len(X_full))
np.random.shuffle(indices)

train_size = int(len(X_full) * 0.8)
train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train, y_train = X_full[train_idx], y_full[train_idx]
X_test, y_test = X_full[test_idx], y_full[test_idx]

# 5. Thêm cột bias (cột toàn 1) vào ma trận X
X_train_b = np.hstack([np.ones((X_train.shape[0], 1)), X_train])
X_test_b = np.hstack([np.ones((X_test.shape[0], 1)), X_test])

# 6. Tính nghiệm OLS bằng NumPy thuần
beta_1 = np.linalg.inv(X_train_b.T @ X_train_b) @ X_train_b.T @ y_train

# 7. Dự đoán và đánh giá
y_pred_1 = X_test_b @ beta_1
mse_1 = mean_squared_error(y_test, y_pred_1)
r2_1 = r2_score(y_test, y_pred_1)

print('=' * 10 + ' ASSIGNMENT 1: OLS MANUAL ' + '=' * 10)
print(f'MSE: {mse_1:.6f}')
print(f'R2 Score: {r2_1:.6f}')

### Assignment 2 (3 scores):

- Standardize the data so that their mean is $0$ and their variance is $1$.
- Compare the results of Linear Regression model when being trained on the original and standardized data.

#### 2.1 Phân tích kỹ thuật & Lưu ý quan trọng:
1. **Chống Data Leakage:** Chỉ tính $\mu_{train}$ và $\sigma_{train}$ trên tập Train, sau đó áp dụng vào cả Train và Test.
2. **Thứ tự thêm cột Bias:** Chuẩn hóa các cột đặc trưng trước, **sau đó** mới nối thêm cột toàn 1. Nếu nối cột 1 trước khi chuẩn hóa, độ lệch chuẩn của cột hằng số sẽ bằng 0 dẫn đến lỗi chia cho 0.
3. **So sánh:** Về lý thuyết đại số, việc chuẩn hóa tuyến tính không làm đổi không gian con của $X$ nên $MSE$ và $R^2$ giữ nguyên giá trị (sai khác chỉ ở bậc số học làm tròn cực nhỏ $\approx 10^{-14}$), nhưng chuẩn hóa giúp cải thiện tính ổn định số học khi tính nghịch đảo ma trận $(X^T X)^{-1}$.

In [ ]:
# 1. Tính mean và std trên tập Train
mu_train = np.mean(X_train, axis=0)
sigma_train = np.std(X_train, axis=0)
sigma_train[sigma_train == 0] = 1e-15

# 2. Chuẩn hóa Z-Score
X_train_std = (X_train - mu_train) / sigma_train
X_test_std = (X_test - mu_train) / sigma_train

# 3. Thêm cột bias sau khi chuẩn hóa
X_train_std_b = np.hstack([np.ones((X_train_std.shape[0], 1)), X_train_std])
X_test_std_b = np.hstack([np.ones((X_test_std.shape[0], 1)), X_test_std])

# 4. Tính toán OLS
beta_2 = np.linalg.inv(X_train_std_b.T @ X_train_std_b) @ X_train_std_b.T @ y_train
y_pred_2 = X_test_std_b @ beta_2

mse_2 = mean_squared_error(y_test, y_pred_2)
r2_2 = r2_score(y_test, y_pred_2)

print('=' * 10 + ' ASSIGNMENT 2: OLS WITH STANDARDIZATION ' + '=' * 10)
print(f'MSE: {mse_2:.6f}')
print(f'R2 Score: {r2_2:.6f}')
print(f'Độ chênh lệch MSE so với Assignment 1: {abs(mse_2 - mse_1):.10e}')

### Assignment 3 (2 scores):

- Perform feature engineering to remove columns that are linearly dependent (colinear) with other columns in the dataset.
- Evaluate the Linear Regression model when beeing trained on the original data and preprocessed data.

#### 3.1 Phân tích đa cộng tuyến (Collinearity):
- Ta tính ma trận hệ số tương quan Pearson giữa các đặc trưng và trực quan hóa bằng Seaborn Heatmap.
- Nhận thấy chỉ số **DC** (Drought Code) và **DMC** (Duff Moisture Code) có tương quan tuyến tính rất mạnh ($r \approx 0.68$).
- Việc có hai đặc trưng phụ thuộc tuyến tính cao sẽ làm ma trận $(X^T X)$ gần suy biến, phương sai của $\hat{\beta}$ tăng vọt.
- Do đó, ta loại bỏ thuộc tính `DC` để phá vỡ hiện tượng đa cộng tuyến.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Vẽ Heatmap ma trận tương quan
corr_matrix = df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix - Forest Fires')
plt.tight_layout()
plt.show()

# 2. Loại bỏ cột DC
df_filtered = df.drop(columns=['DC'])
X_full_f = df_filtered.drop(columns=['area']).values

# 3. Phân chia tập Train/Test theo cùng seed 42
X_train_f, X_test_f = X_full_f[train_idx], X_full_f[test_idx]

# 4. Chuẩn hóa Z-Score
mu_f = np.mean(X_train_f, axis=0)
sigma_f = np.std(X_train_f, axis=0)
sigma_f[sigma_f == 0] = 1e-15

X_train_f_std = (X_train_f - mu_f) / sigma_f
X_test_f_std = (X_test_f - mu_f) / sigma_f

# 5. Thêm cột bias và huấn luyện OLS
X_train_f_std_b = np.hstack([np.ones((X_train_f_std.shape[0], 1)), X_train_f_std])
X_test_f_std_b = np.hstack([np.ones((X_test_f_std.shape[0], 1)), X_test_f_std])

beta_3 = np.linalg.inv(X_train_f_std_b.T @ X_train_f_std_b) @ X_train_f_std_b.T @ y_train
y_pred_3 = X_test_f_std_b @ beta_3

mse_3 = mean_squared_error(y_test, y_pred_3)
r2_3 = r2_score(y_test, y_pred_3)

print('=' * 10 + ' ASSIGNMENT 3: FEATURE ENGINEERING (DROP DC) ' + '=' * 10)
print(f'MSE: {mse_3:.6f}')
print(f'R2 Score: {r2_3:.6f}')
print(f'MSE giảm được so với Assignment 2: {mse_2 - mse_3:.6f} điểm')

### Assignment 4 (2 scores):

- Implement the Linear Regression model using Machine Learning libraries (Scikit Learn or Skorch).
- Compare the results of Linear Regression model constructed manually and from Machine Learning libraries.

#### 4.1 Đánh giá và so sánh:
- Sử dụng `LinearRegression` từ Scikit-Learn trên tập dữ liệu đã chuẩn hóa và loại bỏ cột `DC`.
- Nhận thấy kết quả MSE và $R^2$ hoàn toàn trùng khớp với kết quả nghiệm OLS tự code bằng NumPy thuần.

In [ ]:
from sklearn.linear_model import LinearRegression

model_sklearn = LinearRegression()
model_sklearn.fit(X_train_f_std, y_train)

y_pred_4 = model_sklearn.predict(X_test_f_std)
mse_4 = mean_squared_error(y_test, y_pred_4)
r2_4 = r2_score(y_test, y_pred_4)

print('=' * 10 + ' ASSIGNMENT 4: SKLEARN BENCHMARK ' + '=' * 10)
print(f'Sklearn MSE: {mse_4:.6f}')
print(f'Sklearn R2:  {r2_4:.6f}')
print(f'Sai khác tuyệt đối về MSE giữa NumPy thuần và Sklearn: {abs(mse_3 - mse_4):.15f}')

### Assignment Bonus: Ứng dụng Mô hình OLS trên Dữ liệu Thực tế Cào từ BTTH1

- **Dữ liệu:** File `data.json` cào từ shop Ariana Grande trong BTTH1 (198 sản phẩm).
- **Bài toán:** Hồi quy dự đoán giá sản phẩm (`price`) dựa trên các đặc trưng trích xuất từ tên sản phẩm (`title`) và danh mục (`type`).
- **Mô hình:** Thuật toán Linear Regression OLS giải tích 100% bằng NumPy thuần.

In [ ]:
import json
from sklearn.metrics import mean_absolute_error

bonus_data_path = 'data/data.json' if os.path.exists('data/data.json') else '../data/data.json'
with open(bonus_data_path, 'r', encoding='utf-8') as f:
    raw_bonus = json.load(f)

X_b_list, y_b_list = [], []
music_types = {'CD', '1LP', '2LP', 'Cassette', '7in', 'Digital'}
clothing_types = {'T-Shirt', "Women's T-Shirt", 'Longsleeve T-Shirt', 'Hoodie (pullover)', 'Hoodie (zip up)', 'Crewneck', 'Sweatpants', 'Tank', "Women's Tank"}

for item in raw_bonus:
    p_str = item.get('price', 'N/A')
    if p_str == 'N/A' or p_str is None:
        continue
    try:
        price = float(p_str)
    except ValueError:
        continue
    title = item.get('title', '')
    ptype = item.get('type', '')
    t_len = len(title)
    w_count = len(title.split())
    is_music = 1.0 if ptype in music_types else 0.0
    is_clothing = 1.0 if ptype in clothing_types else 0.0
    is_bundle = 1.0 if ('bundle' in title.lower() or 'fan pack' in title.lower() or ptype == 'Bundle') else 0.0
    X_b_list.append([t_len, w_count, is_music, is_clothing, is_bundle])
    y_b_list.append(price)

X_b = np.array(X_b_list, dtype=np.float64)
y_b = np.array(y_b_list, dtype=np.float64).reshape(-1, 1)

np.random.seed(42)
b_idx = np.arange(len(X_b))
np.random.shuffle(b_idx)
b_train_sz = int(len(X_b) * 0.8)
b_tr_idx, b_te_idx = b_idx[:b_train_sz], b_idx[b_train_sz:]

X_b_tr, y_b_tr = X_b[b_tr_idx], y_b[b_tr_idx]
X_b_te, y_b_te = X_b[b_te_idx], y_b[b_te_idx]

mu_b = np.mean(X_b_tr, axis=0)
sigma_b = np.std(X_b_tr, axis=0)
sigma_b[sigma_b == 0] = 1e-15

X_b_tr_std = (X_b_tr - mu_b) / sigma_b
X_b_te_std = (X_b_te - mu_b) / sigma_b

X_b_tr_std_bias = np.hstack([np.ones((X_b_tr_std.shape[0], 1)), X_b_tr_std])
X_b_te_std_bias = np.hstack([np.ones((X_b_te_std.shape[0], 1)), X_b_te_std])

beta_bonus = np.linalg.inv(X_b_tr_std_bias.T @ X_b_tr_std_bias) @ X_b_tr_std_bias.T @ y_b_tr
y_b_pred = X_b_te_std_bias @ beta_bonus

mse_b = mean_squared_error(y_b_te, y_b_pred)
mae_b = mean_absolute_error(y_b_te, y_b_pred)
r2_b = r2_score(y_b_te, y_b_pred)

print('=' * 10 + ' BONUS ASSIGNMENT RESULTS ' + '=' * 10)
print(f'MSE:  {mse_b:.4f}')
print(f'RMSE: {np.sqrt(mse_b):.4f}')
print(f'MAE:  {mae_b:.4f}')
print(f'R2:   {r2_b:.4f}')

plt.figure(figsize=(7, 6))
plt.scatter(y_b_te, y_b_pred, color='royalblue', edgecolors='k', alpha=0.7, label='Dự đoán')
lims = [min(y_b_te.min(), y_b_pred.min()), max(y_b_te.max(), y_b_pred.max())]
plt.plot(lims, lims, 'r--', label='Đường lý tưởng (y = x)')
plt.xlabel('Giá thực tế ($)')
plt.ylabel('Giá dự đoán ($)')
plt.title('So sánh Giá thực tế vs Giá dự đoán (Crawled Data Bonus)')
plt.legend()
plt.tight_layout()
plt.show()